#Local Financial LLM

Notebook Structure Overview:

Before we dive in, here's what we're building across 7 phases:

Phase 0 → Runtime check & GPU setup

Phase 1 → Install dependencies

Phase 2 → Dataset creation (synthetic JSONL)

Phase 3 → Load base model + tokenizer

Phase 4 → LoRA fine-tuning with TRL

Phase 5 → Inference & testing

Phase 6 → Save & export adapters

**Phase 0 — Runtime CheckPurpose:**

Colab gives us either a T4, A100, or L4 GPU depending on our plan(Free,Pro...). Here in this cell we confirm what we have, check VRAM, and set the right dtype. A 3B model in 4-bit needs ~2.5GB VRAM.

In [1]:
# ── Phase 0: Runtime & GPU Check
import torch
import subprocess
import os

print("=" * 55)
print("  LOCAL FINANCIAL LLM — ENVIRONMENT CHECK")
print("=" * 55)

# GPU info
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU     : {gpu_name}")
    print(f"  VRAM    : {vram_total:.1f} GB")
else:
    print("  ⚠️  No GPU detected — go to Runtime > Change runtime type > T4 GPU")

print(f"  PyTorch : {torch.__version__}")
print(f"  CUDA    : {torch.version.cuda}")
print("=" * 55)

# Set global dtype based on available VRAM
# T4  = 15GB  → use 4-bit quantization + fp16
# A100 = 40GB → can use bf16 without quantization
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
USE_4BIT = VRAM_GB < 20   # T4 → True, A100 → False
DTYPE    = torch.float16 if USE_4BIT else torch.bfloat16

print(f"\n  Strategy : {'4-bit quantization + fp16' if USE_4BIT else 'bf16 full precision'}")
print(f"  USE_4BIT : {USE_4BIT}")
print(f"  DTYPE    : {DTYPE}")

  LOCAL FINANCIAL LLM — ENVIRONMENT CHECK
  GPU     : Tesla T4
  VRAM    : 15.6 GB
  PyTorch : 2.11.0+cu128
  CUDA    : 12.8

  Strategy : 4-bit quantization + fp16
  USE_4BIT : True
  DTYPE    : torch.float16


**Phase 1 — Install Dependencies**

Purpose: We need four key libraries beyond what Colab pre-installs. transformers handles model loading, peft provides LoRA, trl gives us the SFTTrainer (Supervised Fine-Tuning), and bitsandbytes enables 4-bit quantization on the T4. The -q flag keeps output clean — run time is ~2 minutes.

In [2]:
# ── Phase 1 : Dependencies with version pinning ──────────────────
# Root cause: bitsandbytes==0.43.3 has a Triton dependency issue on Colab's
# Python 3.12 + CUDA 12.x environment. Fix: use 0.42.0 + force reinstall.

import subprocess, sys

# Wipe everything broken first
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y",
    "bitsandbytes", "transformers", "peft", "trl", "accelerate"],
    stderr=subprocess.DEVNULL)

# Install unsloth — it pins its own compatible versions of all dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"])

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "datasets==2.21.0",
    "huggingface_hub",
])

print("✅ Installation complete — NOW RESTART RUNTIME before continuing!")
print("   Runtime → Restart session → then skip Phase 1 and run from Phase 0")

✅ Installation complete — NOW RESTART RUNTIME before continuing!
   Runtime → Restart session → then skip Phase 1 and run from Phase 0


**Phase 2 — Dataset Creation**

Purpose: This is the most important phase for domain specialization. We create a JSONL dataset with three task types our model must master:

- **Sentiment classification** — Bullish / Bearish / Neutral with reasoning
- **Macro-to-gold reasoning** — DXY, real yields, CPI → gold outlook
- **Ticker extraction** — pull structured data from unstructured text



> In production we feed thousands of real news items here. For now we build 30 high-quality synthetic samples that teach the format and reasoning style — we can expand this later with real data scraped via yfinance + BeautifulSoup.




In [3]:
# ── Phase 2: Dataset Creation ──────────────────────────────────────────────────
import json
import os
from pathlib import Path

os.makedirs("data", exist_ok=True)

# ── 30 synthetic training samples across 3 task types ──────────────────────────
samples = [

  # ── Task Type 1: Sentiment Classification (10 samples) ──────────────────────
  {
    "instruction": "Analyze the financial sentiment of the following news headline. Classify as Bullish, Bearish, or Neutral. Provide a brief rationale and extract any tickers or assets mentioned.",
    "input": "Federal Reserve signals pause in rate hikes as inflation cools to 2.9%; S&P 500 futures surge 1.4%.",
    "output": "Sentiment: Bullish.\nAssets: S&P 500 (SPX), US Treasuries.\nRationale: A Fed pause removes the primary headwind for equities. Lower rates reduce the discount rate applied to future earnings, inflating equity valuations. Historically, S&P 500 gains an average of 12% in the 12 months following the first Fed pause."
  },
  {
    "instruction": "Analyze the financial sentiment of the following news headline. Classify as Bullish, Bearish, or Neutral. Provide a brief rationale and extract any tickers or assets mentioned.",
    "input": "NVIDIA reports Q2 earnings: EPS $2.70 vs $2.64 expected; revenue $30.0B vs $28.7B consensus. Data center segment +154% YoY.",
    "output": "Sentiment: Bullish.\nTickers: NVDA.\nRationale: Double beat on EPS and revenue with massive data center growth confirms AI infrastructure demand. The +154% YoY segment growth signals a sustained capex cycle from hyperscalers. Expect gap-up open with potential +8 to +12% price reaction."
  },
  {
    "instruction": "Analyze the financial sentiment of the following news headline. Classify as Bullish, Bearish, or Neutral. Provide a brief rationale and extract any tickers or assets mentioned.",
    "input": "US CPI unexpectedly rises to 4.1% YoY vs 3.7% forecast; core CPI +3.9% MoM.",
    "output": "Sentiment: Bearish.\nAssets: Equities (SPX, NDX), Bonds (TLT). Bullish: XAU/USD, USD.\nRationale: Inflation re-acceleration forces the Fed to maintain restrictive policy longer. This reprices rate cut expectations downward, pressuring growth stocks and long-duration bonds. Gold benefits as an inflation hedge despite USD strength creating a short-term headwind."
  },
  {
    "instruction": "Analyze the financial sentiment of the following news headline. Classify as Bullish, Bearish, or Neutral. Provide a brief rationale and extract any tickers or assets mentioned.",
    "input": "Apple announces $110B share buyback program; dividend raised 4% to $0.25/quarter.",
    "output": "Sentiment: Bullish.\nTickers: AAPL.\nRationale: Largest buyback in Apple's history signals management confidence in cash generation. Share repurchases mechanically reduce float, boosting EPS without revenue growth. The dividend increase signals commitment to returning capital. Expect +3 to +5% price reaction."
  },
  {
    "instruction": "Analyze the financial sentiment of the following news headline. Classify as Bullish, Bearish, or Neutral. Provide a brief rationale and extract any tickers or assets mentioned.",
    "input": "Tesla Q3 deliveries: 435,059 vehicles vs 469,000 analyst consensus; miss of 7.2%.",
    "output": "Sentiment: Bearish.\nTickers: TSLA.\nRationale: A 7.2% delivery miss indicates demand softness or production constraints. Delivery volume is Tesla's primary near-term KPI — consensus misses historically trigger -5 to -10% reactions. Margin pressure from price cuts compounds the bearish signal."
  },
  {
    "instruction": "Analyze the financial sentiment of the following news headline. Classify as Bullish, Bearish, or Neutral. Provide a brief rationale and extract any tickers or assets mentioned.",
    "input": "US unemployment rate holds steady at 3.9%; nonfarm payrolls add 187,000 jobs vs 185,000 expected.",
    "output": "Sentiment: Neutral-to-Bullish.\nAssets: USD, SPX.\nRationale: A near-consensus payroll print with stable unemployment is a Goldilocks signal — strong enough to avoid recession fears, weak enough to not force additional Fed tightening. Likely muted market reaction with slight USD strength."
  },
  {
    "instruction": "Analyze the financial sentiment of the following news headline. Classify as Bullish, Bearish, or Neutral. Provide a brief rationale and extract any tickers or assets mentioned.",
    "input": "China manufacturing PMI falls to 48.1, contracting for third consecutive month; exports -14.5% YoY.",
    "output": "Sentiment: Bearish.\nAssets: Copper (HG), Iron Ore, AUD/USD, Emerging Market ETFs (EEM).\nRationale: Sustained Chinese manufacturing contraction signals weakening global industrial demand. Copper and iron ore — China-sensitive commodities — face downward pressure. AUD/USD declines as Australia's commodity exports depend heavily on Chinese demand."
  },
  {
    "instruction": "Analyze the financial sentiment of the following news headline. Classify as Bullish, Bearish, or Neutral. Provide a brief rationale and extract any tickers or assets mentioned.",
    "input": "JPMorgan Chase reports record Q2 profit of $18.1B; NII $22.9B, up 44% YoY.",
    "output": "Sentiment: Bullish.\nTickers: JPM. Positive read-through: BAC, WFC, GS, MS.\nRationale: Record profits driven by net interest income expansion in a high-rate environment. Strong JPM results typically lift the entire banking sector. 44% NII growth confirms that rate hikes have disproportionately benefited large-cap banks."
  },
  {
    "instruction": "Analyze the financial sentiment of the following news headline. Classify as Bullish, Bearish, or Neutral. Provide a brief rationale and extract any tickers or assets mentioned.",
    "input": "SEC charges Binance and Coinbase with operating unregistered securities exchanges.",
    "output": "Sentiment: Bearish.\nAssets: BTC, ETH, BNB, crypto sector broadly.\nRationale: Regulatory action against the two largest crypto exchanges creates existential uncertainty for the sector. Forced delisting of tokens classified as securities would destroy liquidity. Expect -10 to -20% drawdown in BTC/ETH as contagion fear spreads."
  },
  {
    "instruction": "Analyze the financial sentiment of the following news headline. Classify as Bullish, Bearish, or Neutral. Provide a brief rationale and extract any tickers or assets mentioned.",
    "input": "Oil prices drop 4.2% as OPEC+ delays production cut extension; Saudi Arabia signals output flexibility.",
    "output": "Sentiment: Bearish (energy sector). Mixed for broader market.\nAssets: WTI Crude (CL), XOM, CVX, BP. Bullish: Consumer discretionary, airlines (DAL, UAL).\nRationale: Lower oil prices compress energy sector margins but reduce input costs for transportation, manufacturing, and consumers. Net effect on SPX is mild positive due to CPI relief narrative."
  },

  # ── Task Type 2: Gold-Macro Reasoning (10 samples) ──────────────────────────
  {
    "instruction": "Given the macroeconomic data snapshot below, predict the directional 30-day outlook for gold (XAU/USD) and explain the three key driving factors. Rate your conviction as High, Medium, or Low.",
    "input": "DXY: 106.8 (+1.2% WoW). US 10Y Real Yield: 2.4%. CPI YoY: 3.2%. VIX: 14.1. Fed Funds Rate: 5.50%. SPX: near all-time highs.",
    "output": "Outlook: Bearish on gold (30-day). Conviction: High.\n\nFactor 1 — DXY Strength (106.8): Gold has a historically strong inverse correlation with the USD (r = -0.75 over 20 years). A DXY above 105 historically suppresses gold by 0.8–1.1% per 1-point DXY gain.\n\nFactor 2 — Real Yields at 2.4%: This is the most important driver. Gold pays no yield. At 2.4% real yield, the opportunity cost of holding gold versus TIPS is significant. Gold tends to underperform when real yields exceed 1.5%.\n\nFactor 3 — Low VIX (14.1) + SPX at highs: Minimal fear in the market suppresses safe-haven demand. Investors rotate into equities, not gold.\n\nPrice target range: $1,850–$1,920 (from ~$2,000 baseline). Key upside risk: Any geopolitical shock or Fed pivot language."
  },
  {
    "instruction": "Given the macroeconomic data snapshot below, predict the directional 30-day outlook for gold (XAU/USD) and explain the three key driving factors. Rate your conviction as High, Medium, or Low.",
    "input": "DXY: 99.2 (-1.8% WoW). US 10Y Real Yield: 0.3%. CPI YoY: 5.1%. VIX: 28.4. Fed Funds Rate: 2.25%. Geopolitical event: escalation in Middle East.",
    "output": "Outlook: Strongly Bullish on gold (30-day). Conviction: High.\n\nFactor 1 — DXY Weakness (99.2): Below 100 DXY is a historically strong tailwind for gold. Combined with -1.8% weekly decline, momentum is building. Each 1% DXY decline historically adds ~$15–20 to gold.\n\nFactor 2 — Near-Zero Real Yield (0.3%): Opportunity cost of holding gold approaches zero. This is the ideal environment for gold accumulation — inflation at 5.1% with rates at 2.25% means deeply negative real rates in real terms, historically gold's strongest environment.\n\nFactor 3 — VIX at 28.4 + Geopolitical Risk: VIX above 25 triggers institutional safe-haven flows into gold. Geopolitical escalation historically adds a 3–8% risk premium to gold within 2 weeks.\n\nPrice target range: $2,150–$2,300. Key downside risk: Sudden ceasefire or surprise Fed hawkishness."
  },
  {
    "instruction": "Given the macroeconomic data snapshot below, predict the directional 30-day outlook for gold (XAU/USD) and explain the three key driving factors. Rate your conviction as High, Medium, or Low.",
    "input": "DXY: 102.5 (flat WoW). US 10Y Real Yield: 1.1%. CPI YoY: 2.8%. VIX: 18.7. Central bank gold purchases: +387 tonnes YTD (record pace).",
    "output": "Outlook: Moderately Bullish on gold (30-day). Conviction: Medium.\n\nFactor 1 — Neutral DXY (102.5): No significant USD headwind or tailwind. Gold's direction will be determined by other factors.\n\nFactor 2 — Real Yield at 1.1%: Above zero, creating modest opportunity cost, but not at the 2%+ level that historically crushes gold. A neutral factor with slight downward bias.\n\nFactor 3 — Record Central Bank Purchases (+387T YTD): This is the dominant bullish signal. Central bank demand (particularly from China, India, Turkey) absorbs supply and signals long-term de-dollarization trends. This structural bid provides a price floor and offsets the real yield headwind.\n\nPrice target range: $2,050–$2,150. The central bank demand factor gives this a positive skew despite neutral macro readings."
  },
  {
    "instruction": "Given the macroeconomic data snapshot below, predict the directional 30-day outlook for gold (XAU/USD) and explain the three key driving factors. Rate your conviction as High, Medium, or Low.",
    "input": "DXY: 101.0 (-0.5%). US 10Y Real Yield: 1.8%. CPI YoY: 3.0%. VIX: 16.2. US debt ceiling crisis ongoing. Credit rating downgrade risk elevated.",
    "output": "Outlook: Bullish on gold (30-day). Conviction: Medium-High.\n\nFactor 1 — US Fiscal Risk (Debt Ceiling + Downgrade): US sovereign credit risk directly undermines USD confidence. The 2011 US downgrade triggered a 20% gold rally in 2 months. Current debt ceiling uncertainty mirrors that setup.\n\nFactor 2 — Mild DXY Weakness (101.0): Slightly below neutral territory. If debt ceiling fears accelerate dollar selling, DXY could break below 100 — a strongly bullish trigger for gold.\n\nFactor 3 — Real Yield at 1.8% (bearish offset): This is the primary headwind. However, fiscal risk narratives tend to override rate-based models in the short term as investors question the 'risk-free' status of US Treasuries.\n\nNet assessment: Fiscal tail risk outweighs real yield headwind in this specific macro setup. Price target: $2,100–$2,200."
  },
  {
    "instruction": "Given the macroeconomic data snapshot below, predict the directional 30-day outlook for gold (XAU/USD) and explain the three key driving factors. Rate your conviction as High, Medium, or Low.",
    "input": "DXY: 103.4. US 10Y Real Yield: 2.0%. CPI YoY: 2.3% (approaching Fed target). VIX: 12.8. Fed signals rate cuts starting next quarter.",
    "output": "Outlook: Bullish on gold (30-day). Conviction: High.\n\nFactor 1 — Imminent Fed Rate Cuts: This is the single most powerful catalyst for gold. Rate cut cycles reduce the opportunity cost of holding gold and typically weaken the USD. Gold has rallied an average of 18% in the 12 months following the first cut since 1990.\n\nFactor 2 — CPI at 2.3% (near target): Inflation control removes the Fed's constraint to cut. The path to easing is now clear, which markets reprice immediately — forward rates drop, DXY weakens, gold benefits.\n\nFactor 3 — DXY at 103.4 (likely to fall): As rate cuts materialize, yield differential narrows between USD and other currencies. DXY typically falls 3–5% in the 6 months after first cut, providing additional gold tailwind.\n\nPrice target: $2,200–$2,400. The rate-cut-cycle setup is the highest-conviction gold bull scenario."
  },
  {
    "instruction": "Explain the historical relationship between the US Dollar Index (DXY) and gold prices (XAU/USD). Include the correlation coefficient, key exceptions, and how a trader should weight this relationship in a multi-factor model.",
    "input": "",
    "output": "DXY-Gold Relationship:\n\nCorrelation: The DXY and XAU/USD have a historically inverse correlation of approximately -0.70 to -0.80 over rolling 20-year periods. This means: when DXY rises 1%, gold tends to fall 0.7–0.8%.\n\nMechanism: Gold is priced in USD globally. A stronger dollar makes gold more expensive for non-USD buyers, reducing demand. Conversely, USD weakness makes gold cheaper internationally, boosting demand.\n\nKey Exceptions (when correlation breaks down):\n1. Geopolitical black swan events — both USD and gold can rally simultaneously as safe havens (2022 Russia-Ukraine: DXY +15%, Gold +8% initially).\n2. Stagflation environments — gold can rise despite USD strength when inflation exceeds nominal yields.\n3. Central bank demand shocks — structural buying can override short-term USD correlation.\n\nMulti-factor model weighting recommendation:\n- DXY direction: 25% weight\n- US 10Y Real Yield: 35% weight (most predictive single factor)\n- VIX/risk sentiment: 20% weight\n- Central bank demand: 15% weight\n- Geopolitical premium: 5% (binary, event-driven)\n\nTrader rule of thumb: Real yields matter more than nominal yields. Focus on TIPS (Treasury Inflation-Protected Securities) 10Y yield as the primary gold signal."
  },
  {
    "instruction": "A trader holds a long gold position. Current XAU/USD: $2,050. The following macro events are scheduled this week. Assess the net risk to the position.",
    "input": "Monday: Fed Chair speech (expected hawkish tone). Wednesday: US CPI release (consensus 3.3%, prior 3.5%). Friday: US Nonfarm Payrolls (consensus 180K, prior 206K).",
    "output": "Net Risk Assessment: Moderate Bearish risk to long gold position this week.\n\nMonday — Fed Chair Speech (Hawkish):\nRisk: High negative impact. Hawkish language reinforces 'higher for longer' narrative, pushes real yields up, gold down. Estimated impact: -0.8% to -1.5% on gold if language surprises hawkishly. Action: Consider reducing position size 10–15% before the speech as a hedge.\n\nWednesday — CPI (3.3% expected vs 3.5% prior):\nRisk: Conditional. If CPI comes in at 3.3% or below → Bullish for gold (disinflation narrative supports cut expectations). If CPI surprises above 3.4% → Bearish (delays cuts, strengthens USD). Probability distribution: 60% in-line/below, 40% above. Net expected value: slight negative given post-speech setup.\n\nFriday — NFP (180K expected vs 206K prior):\nRisk: Low-to-moderate. Payroll deceleration expected. If in-line, neutral for gold. If significantly below 150K → Bullish (recession fears drive safe-haven demand). If above 220K → Bearish.\n\nWeek summary: The sequencing matters. A hawkish Fed Monday sets a bearish tone that makes Wednesday CPI and Friday NFP higher-risk events. Recommend: reduce position 15–20% before Monday, re-evaluate after CPI."
  },
  {
    "instruction": "What is the relationship between US 10-year real yields and gold prices? Explain the mechanism and provide historical examples.",
    "input": "",
    "output": "Real Yields and Gold — The Primary Relationship:\n\nDefinition: Real yield = Nominal 10Y Treasury yield minus expected inflation (proxied by 10Y breakeven inflation rate). Tracked via TIPS (Treasury Inflation-Protected Securities).\n\nMechanism: Gold is a zero-yield asset. When real yields are high (e.g., 2%), investors earn 2% risk-free in TIPS — making gold's zero yield unattractive by comparison. When real yields are negative (e.g., -1%), holding cash/bonds means losing real purchasing power — making gold's 'zero yield' actually superior.\n\nHistorical examples:\n1. 2020–2021 (COVID era): Real yields fell to -1.0% → Gold surged from $1,500 to $2,075 (+38%).\n2. 2022 rate hikes: Real yields rose from -1% to +2% in 12 months → Gold fell from $2,050 to $1,620 (-21%).\n3. 2024–2025: Real yields at 2%+ yet gold at all-time highs — the central bank demand anomaly. This is a structural break from historical patterns driven by de-dollarization demand.\n\nTrading rule: Gold tends to perform best when real yields are below 0.5%. Above 1.5% real yields, gold faces significant headwinds unless a geopolitical or structural factor overrides. Monitor the 10Y TIPS yield at: FRED series DFII10."
  },
  {
    "instruction": "Describe how interest rate differentials between countries affect gold prices indirectly through currency channels.",
    "input": "US Fed Funds Rate: 5.50%. ECB Rate: 4.50%. Bank of Japan Rate: 0.10%. Bank of England Rate: 5.25%.",
    "output": "Interest Rate Differentials and Gold — The Currency Channel:\n\nMechanism: Interest rate differentials drive currency flows. Higher rates attract capital → strengthen currency. Since gold is priced in USD, currency shifts affect gold demand from non-USD buyers.\n\nCurrent setup analysis:\n\n1. USD vs EUR (spread: +100bps): USD yield advantage attracts capital into dollars, strengthening DXY → modest headwind for gold. However, spread is narrowing as ECB catches up — a narrowing spread is dollar-negative and gold-positive.\n\n2. USD vs JPY (spread: +540bps): This is the most extreme differential globally. Yen weakness (USD/JPY high) forces the Bank of Japan toward eventual normalization. When BOJ raises rates (any move from 0.10%), the unwinding of yen carry trades could be massive — potentially weakening USD and strongly supporting gold.\n\n3. USD vs GBP (spread: +25bps): Near-zero differential. GBP-based gold buyers face minimal currency friction — neutral effect.\n\nKey watch: BOJ normalization is arguably the most underpriced catalyst for gold in 2025. The yen carry trade (estimated $4T in size) unwind would force dollar selling at scale, with gold as a primary beneficiary. Monitor BOJ meeting dates."
  },
  {
    "instruction": "Explain the concept of the 'Gold-Oil Ratio' and how it can signal economic turning points.",
    "input": "Current Gold price: $2,100/oz. Current WTI Crude: $72/barrel. Gold-Oil ratio: 29.2. Historical average: 15–20.",
    "output": "Gold-Oil Ratio Analysis:\n\nDefinition: Gold-Oil Ratio = Gold price ($/oz) ÷ WTI Crude price ($/barrel). Measures how many barrels of oil one ounce of gold can buy.\n\nCurrent reading: 29.2 (significantly above historical average of 15–20).\n\nInterpretation: A ratio of 29.2 means gold is historically expensive relative to oil, OR oil is historically cheap relative to gold. This divergence signals one of three scenarios:\n1. Economic slowdown — oil demand is depressed due to weakening global growth while gold benefits from safe-haven flows. Most likely interpretation.\n2. Oil supply glut — OPEC oversupply temporarily depresses oil prices independent of demand.\n3. Gold overvaluation — gold has run ahead of fundamentals (less likely given central bank demand).\n\nHistorical context:\n- Ratio peaked at ~35 during COVID crash (March 2020) → oil collapsed, gold held.\n- Ratio at 15–16 during 2011 gold peak + supply-constrained oil environment.\n- Ratio above 25 has historically preceded oil price rebounds as the ratio mean-reverts.\n\nTrading signal: Current ratio of 29.2 suggests: (a) consider long oil / short gold as a ratio trade, or (b) interpret high ratio as recession warning — overweight gold defensively while underweighting cyclical commodities."
  },

  # ── Task Type 3: Ticker Extraction & Structured Analysis (10 samples) ─────────
  {
    "instruction": "Extract all financial entities from the text below. Return: (1) Stock tickers with company names, (2) Commodities, (3) Macro indicators, (4) Overall market sentiment score from -5 (extreme bear) to +5 (extreme bull).",
    "input": "Berkshire Hathaway increased its Apple stake to $174B while reducing holdings in HP and Activision. Warren Buffett also disclosed a new position in Occidental Petroleum. Meanwhile, gold hit $2,150 as the dollar weakened.",
    "output": "Stock Tickers:\n- AAPL (Apple Inc.) — Berkshire increased position, Bullish signal (Buffett conviction buy)\n- HP (HP Inc.) — Berkshire reducing, mild Bearish signal\n- ATVI (Activision Blizzard) — Berkshire reducing, Bearish/exiting\n- OXY (Occidental Petroleum) — New Berkshire position, Bullish signal\n- BRK.B / BRK.A (Berkshire Hathaway) — The disclosing entity\n\nCommodities:\n- XAU/USD (Gold) — Price: $2,150. Driver: USD weakness. Sentiment: Bullish\n\nMacro indicators:\n- USD Index (DXY) — Weakening. Implied Bearish for USD, Bullish for gold and multinationals\n\nSentiment Score: +3 (Moderately Bullish)\nRationale: Buffett's moves are high-conviction signals. New OXY position and increased AAPL suggest risk-on appetite from the world's most followed value investor. Gold strength confirms macro uncertainty remains elevated."
  },
  {
    "instruction": "Extract all financial entities from the text below. Return: (1) Stock tickers with company names, (2) Commodities, (3) Macro indicators, (4) Overall market sentiment score from -5 (extreme bear) to +5 (extreme bull).",
    "input": "The yield curve inverted further today with the 2Y-10Y spread reaching -89bps. Bank stocks led declines with JPMorgan, Bank of America, and Wells Fargo all falling over 2%. Regional banks (KRE ETF) dropped 4.1%.",
    "output": "Stock Tickers:\n- JPM (JPMorgan Chase) — Down >2%, yield curve inversion pressure\n- BAC (Bank of America) — Down >2%, same driver\n- WFC (Wells Fargo) — Down >2%, same driver\n- KRE (SPDR S&P Regional Banking ETF) — Down 4.1%, amplified pain in regional banks\n\nCommodities: None mentioned.\n\nMacro Indicators:\n- 2Y Treasury Yield — Rising (implied, causing inversion)\n- 10Y Treasury Yield — Below 2Y yield\n- 2Y-10Y Yield Spread: -89bps (deeply inverted — historically precedes recession by 12–18 months)\n\nSentiment Score: -3 (Moderately Bearish)\nRationale: Yield curve inversion at -89bps is one of the most reliable recession predictors (accuracy ~80% in post-1970 data). Bank NII margins compress with an inverted curve. Regional banks face the most acute pressure due to higher sensitivity to funding costs."
  },
  {
    "instruction": "Extract all financial entities from the text below. Return: (1) Stock tickers with company names, (2) Commodities, (3) Macro indicators, (4) Overall market sentiment score from -5 (extreme bear) to +5 (extreme bull).",
    "input": "TSMC reported strong Q3 results with revenue NT$759.7B (+36% YoY), driven by AI chip demand. CoWoS packaging capacity sold out through 2025. AMD and Intel customers competing for allocation. Taiwan dollar strengthened 1.2%.",
    "output": "Stock Tickers:\n- TSM (TSMC) — Strong results, +36% YoY revenue. Bullish. Primary beneficiary.\n- AMD (Advanced Micro Devices) — Customer competing for TSMC capacity. Positive read-through: strong AI chip demand, but supply-constrained. Bullish.\n- INTC (Intel) — Also competing for capacity. Neutral-to-Positive.\n- NVDA (Nvidia, implied) — CoWoS packaging primarily serves Nvidia H100/H200 dies. Strongly Bullish by implication.\n\nCommodities: None explicitly.\n\nMacro Indicators:\n- TWD/USD (Taiwan Dollar) — Strengthened 1.2%. Signals capital inflows into Taiwan tech sector.\n\nSentiment Score: +4 (Bullish)\nRationale: TSMC results are the gold standard for AI semiconductor demand visibility. Sold-out CoWoS through 2025 confirms multi-year AI infrastructure cycle. Strong positive read-through for the entire AI chip supply chain: NVDA, AMD, AMAT, ASML."
  },
  {
    "instruction": "From the following earnings call transcript excerpt, extract all quantitative data points, classify the tone as Beat/Miss/In-Line, and identify the 3 most important sentences for an investor.",
    "input": "CEO: 'We delivered revenue of $8.2 billion, growing 22% year-over-year, ahead of our guidance of $7.9 to $8.0 billion. Gross margins came in at 71.3%, up 180 basis points sequentially. However, we are guiding next quarter to $8.0 to $8.3 billion, which is below the Street's $8.6 billion estimate. We continue to see headwinds in our legacy hardware segment, which declined 12% year-over-year.'",
    "output": "Quantitative Data Points:\n- Revenue: $8.2B (actual) vs $7.9–8.0B (guidance) vs implied consensus: Beat\n- Revenue growth: +22% YoY\n- Gross margin: 71.3% (+180bps QoQ)\n- Q next guidance: $8.0–8.3B vs $8.6B Street consensus: MISS (-3.5% to -7% below)\n- Legacy hardware segment: -12% YoY\n\nClassification: Mixed — Current quarter Beat, Forward guidance Miss\n\nTop 3 Most Important Sentences for Investors:\n1. 'We are guiding next quarter to $8.0–8.3 billion, which is below the Street's $8.6 billion estimate.' → Forward guidance miss is the primary near-term negative catalyst.\n2. 'Gross margins came in at 71.3%, up 180 basis points sequentially.' → Margin expansion is the primary positive signal — quality of earnings improving.\n3. 'Legacy hardware segment declined 12% year-over-year.' → Secular decline in legacy business; investors need to assess whether core growth can offset this drag.\n\nLikely market reaction: -2% to -4% initial reaction on guidance miss, partially offset by strong current-quarter beat and margin expansion."
  },
  {
    "instruction": "Extract all financial entities from the text below. Return: (1) Stock tickers with company names, (2) Commodities, (3) Macro indicators, (4) Overall market sentiment score from -5 (extreme bear) to +5 (extreme bull).",
    "input": "Crude oil fell 6% this week after EIA reported a surprise inventory build of 8.4 million barrels vs an expected draw of 1.2 million. WTI touched $68/bbl. Energy names Exxon, Chevron, and Halliburton all underperformed. Natural gas (Henry Hub) also dropped 3%.",
    "output": "Stock Tickers:\n- XOM (Exxon Mobil) — Underperforming, Bearish near-term\n- CVX (Chevron) — Underperforming, Bearish near-term\n- HAL (Halliburton) — Underperforming, Bearish near-term (oilfield services most sensitive to rig count)\n\nCommodities:\n- WTI Crude (CL1!) — Price: $68/bbl, -6% weekly. Driver: inventory build. Bearish.\n- Natural Gas (Henry Hub / NG1!) — Down 3%. Bearish.\n\nMacro Indicators:\n- EIA Crude Inventory: +8.4M barrels (vs -1.2M expected) — 9.6M barrel surprise. Indicates demand weakness or supply glut.\n\nSentiment Score: -2 (Mildly Bearish — energy sector specific)\nBroader market: Neutral-to-Positive. Lower oil = lower gasoline prices = consumer spending relief = mild CPI benefit. Airlines (DAL, UAL) and consumer discretionary benefit."
  },
  {
    "instruction": "Extract all financial entities from the text below. Return: (1) Stock tickers with company names, (2) Commodities, (3) Macro indicators, (4) Overall market sentiment score from -5 (extreme bear) to +5 (extreme bull).",
    "input": "The European Central Bank raised rates by 25bps to 4.50%, its 10th consecutive hike. EUR/USD fell to 1.0650 as the ECB hinted this may be the final hike. German Bund yields rose 8bps to 2.89%. European bank stocks (SAN, BNP, DB) rallied 1.5–2% on NII expansion hopes.",
    "output": "Stock Tickers:\n- SAN (Banco Santander) — Rallied ~1.5–2%. Bullish: higher rates expand net interest income.\n- BNP (BNP Paribas, OTC: BNPQY) — Same driver. Bullish.\n- DB (Deutsche Bank) — Same driver. Bullish.\n\nCommodities: None explicitly mentioned.\n\nMacro Indicators:\n- ECB Rate: 4.50% (+25bps) — 10th consecutive hike. Potential terminal rate.\n- EUR/USD: 1.0650 (falling on dovish forward guidance — 'final hike' signal weakens EUR).\n- German 10Y Bund Yield: 2.89% (+8bps)\n\nSentiment Score: +1 (Mildly Bullish — European banking sector specific)\nRationale: Terminal rate signal is mixed — bullish for EUR banks (NII peak) but negative for EUR as carry appeal diminishes. ECB pause would allow bond prices to recover. Broader European equities neutral."
  },
  {
    "instruction": "Extract all financial entities from the text below. Return: (1) Stock tickers with company names, (2) Commodities, (3) Macro indicators, (4) Overall market sentiment score from -5 (extreme bear) to +5 (extreme bull).",
    "input": "Palantir Technologies beat Q3 EPS estimates by 40% and raised full-year guidance. US government contract wins accelerated to $323M. The company achieved GAAP profitability for the fifth consecutive quarter. Stock up 18% after hours.",
    "output": "Stock Tickers:\n- PLTR (Palantir Technologies) — +18% AH. Strong Beat + raised guidance + GAAP profitability milestone.\n\nCommodities: None.\n\nMacro Indicators: None directly. Sector read-through: defense tech / AI government contracts accelerating.\n\nSentiment Score: +4 (Bullish on PLTR specifically)\nKey factors:\n1. 40% EPS beat is exceptional — management either guided conservatively or execution significantly outperformed.\n2. US government contract acceleration ($323M) confirms PLTR's AI platform is embedded in critical defense infrastructure — high switching cost moat.\n3. Fifth consecutive GAAP profitable quarter removes the primary bear case (profitability concerns). This is a narrative inflection point.\n\nRead-through: Positive for AI government contract space broadly. Watch: BBAI (BigBear.ai), CACI, Booz Allen Hamilton (BAH)."
  },
  {
    "instruction": "Extract all financial entities from the text below. Return: (1) Stock tickers with company names, (2) Commodities, (3) Macro indicators, (4) Overall market sentiment score from -5 (extreme bear) to +5 (extreme bull).",
    "input": "Bitcoin surged 12% to $68,500 after BlackRock's spot Bitcoin ETF (IBIT) reported record daily inflows of $788M. Coinbase (COIN) and MicroStrategy (MSTR) both gained over 10% in sympathy. Ethereum rose 8% to $3,450.",
    "output": "Stock Tickers:\n- COIN (Coinbase) — +10%, benefiting from increased crypto trading volume. Bullish.\n- MSTR (MicroStrategy) — +10%, highly leveraged Bitcoin proxy. Bullish (amplified BTC move).\n- BLK (BlackRock, issuer of IBIT ETF) — Mild positive, AUM growth signal.\n\nCrypto Assets:\n- BTC (Bitcoin) — $68,500, +12%. Driver: Institutional ETF inflows ($788M single day). Bullish.\n- ETH (Ethereum) — $3,450, +8%. Beta trade following BTC. Bullish.\n- IBIT (BlackRock Bitcoin ETF) — Record $788M daily inflow. Strong institutional adoption signal.\n\nMacro Indicators: Implied — risk appetite high, liquidity conditions favorable.\n\nSentiment Score: +4 (Bullish on crypto sector)\nRationale: Institutional ETF inflows of $788M in a single day represent a structural demand shift, not retail speculation. BlackRock's distribution network brings crypto to pension funds, RIAs, and wealth managers — a durable new buyer class. This is not a short-term catalyst."
  },
  {
    "instruction": "From the earnings call excerpt, identify forward guidance, beat/miss status, and the primary risk the CFO is signaling.",
    "input": "CFO: 'While we're proud of our Q3 performance with $3.1B in revenue, representing 8% growth, we are revising our full-year guidance from $12.4B to $12.1B due to macroeconomic headwinds in our EMEA segment. Foreign exchange headwinds will create an approximately 200bps drag on margins in Q4. We expect these conditions to persist through at least H1 next year.'",
    "output": "Beat/Miss Status: Revenue beat implied (Q3 $3.1B at 8% growth). Forward guidance CUT — negative.\n\nGuidance Revision:\n- Previous full-year: $12.4B\n- Revised full-year: $12.1B\n- Cut magnitude: -$300M (-2.4% reduction)\n- Reason: EMEA segment macroeconomic headwinds\n\nPrimary Risk Signaled by CFO:\n1. EMEA demand weakness — geographic concentration risk. Company has meaningful Europe/Middle East/Africa exposure that is underperforming.\n2. FX headwind (-200bps margin impact in Q4) — significant. 200bps margin drag translates directly to lower EPS. If Q4 margins were expected at 25%, this implies actual margins of ~23%.\n3. Duration of headwinds ('through at least H1 next year') — the word 'at least' is critical. Management is signaling uncertainty about recovery timing — the actual downturn may extend beyond H1.\n\nInvestor takeaway: Guidance cut + extended headwind duration language is the double negative. Expect -5 to -8% reaction. Watch for analyst estimate revision cascade."
  },
  {
    "instruction": "Extract all financial entities from the text below. Return: (1) Stock tickers with company names, (2) Commodities, (3) Macro indicators, (4) Overall market sentiment score from -5 (extreme bear) to +5 (extreme bull).",
    "input": "Silver (XAG/USD) surged 4.2% to $29.40 as industrial demand data from China showed a rebound. The Gold-Silver ratio compressed from 85 to 81. Mining stocks First Majestic (AG) and Pan American Silver (PAAS) both jumped over 6%.",
    "output": "Stock Tickers:\n- AG (First Majestic Silver) — +6%+. Pure-play silver miner. Bullish. High beta to silver price.\n- PAAS (Pan American Silver) — +6%+. Diversified silver/gold miner. Bullish.\n\nCommodities:\n- XAG/USD (Silver) — $29.40, +4.2%. Driver: China industrial demand rebound.\n- XAU/USD (Gold, implied) — Gold-Silver ratio compression from 85→81 implies gold was relatively flat while silver outperformed.\n- Gold-Silver Ratio: 81 (down from 85). Tightening ratio = silver outperformance = risk-on signal.\n\nMacro Indicators:\n- China industrial demand — Rebounding. Bullish for industrial metals broadly (copper, silver, platinum).\n\nSentiment Score: +3 (Moderately Bullish — precious/industrial metals)\nRationale: Silver is a hybrid asset — 50% investment demand (like gold), 50% industrial demand (solar panels, EVs, electronics). A China demand rebound activates the industrial half of silver's demand equation. Gold-Silver ratio compression from 85→81 historically signals the start of a silver outperformance cycle. Target ratio: 70–75 if industrial recovery sustains."
  }
]

# Write to JSONL
output_path = "data/financial_train.jsonl"
with open(output_path, "w") as f:
    for sample in samples:
        f.write(json.dumps(sample) + "\n")

print(f"✅ Dataset created: {len(samples)} samples")
print(f"   Saved to: {output_path}")
print(f"\n   Task breakdown:")
print(f"   • Sentiment classification : 10 samples")
print(f"   • Gold-macro reasoning     : 10 samples")
print(f"   • Ticker extraction/QA     : 10 samples")
print(f"\n   Sample preview:")
print(json.dumps(samples[0], indent=2)[:400] + "...")

✅ Dataset created: 30 samples
   Saved to: data/financial_train.jsonl

   Task breakdown:
   • Sentiment classification : 10 samples
   • Gold-macro reasoning     : 10 samples
   • Ticker extraction/QA     : 10 samples

   Sample preview:
{
  "instruction": "Analyze the financial sentiment of the following news headline. Classify as Bullish, Bearish, or Neutral. Provide a brief rationale and extract any tickers or assets mentioned.",
  "input": "Federal Reserve signals pause in rate hikes as inflation cools to 2.9%; S&P 500 futures surge 1.4%.",
  "output": "Sentiment: Bullish.\nAssets: S&P 500 (SPX), US Treasuries.\nRationale: A F...


**Phase 3 — Load Base Model & Tokenizer**

Purpose: We load Qwen2.5-3B-Instruct with 4-bit NF4 quantization. This is the configuration that lets a 3B parameter model (which would normally need ~6GB in float16) fit comfortably in a T4's 15GB VRAM. We then format our dataset into the model's native chat template — this matters because Qwen2.5 was trained with specific <|im_start|> tokens and the model will perform significantly worse if  we a different format.


In [4]:
# ── Phase 3: Load Base Model & Tokenizer ──────────────────────────────────────
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import Dataset

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"


print(f"Loading model: {MODEL_ID}")
print("This will download ~2GB on first run — cached on subsequent runs.\n")

# ── 4-bit quantization config ──────────────────────────────────────────────────
# NF4 (NormalFloat4) is better than int4 for LLMs — preserves distribution shape
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,  # nested quantization saves extra 0.4bpw
)

# ── Load tokenizer ─────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # required for SFTTrainer

# ── Load model ─────────────────────────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False               # disable KV cache during training
model.config.pretraining_tp = 1             # disable tensor parallelism

print(f"\n✅ Model loaded.")
print(f"   Parameters  : ~3B")
print(f"   VRAM usage  : ~2.5GB (NF4 4-bit)")

# ── Format dataset into chat template ─────────────────────────────────────────
def format_sample(sample: dict) -> dict:
    """
    Convert our instruction-input-response format into Qwen2.5's native chat format.
    The model was trained with <|im_start|> tokens — using this exact format
    is critical for performance. Do not change this if using Qwen2.5.
    """
    system_msg = (
        "You are a specialized financial analyst with deep expertise in "
        "stock market sentiment analysis, ticker extraction, earnings interpretation, "
        "and gold-macroeconomic relationships including DXY, real yields, CPI, and central bank dynamics."
    )

    user_content = sample["instruction"]
    if sample.get("input", "").strip():
        user_content += f"\n\n{sample['input']}"

    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_content},
        {"role": "assistant", "content": sample["output"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

# Load and format dataset
raw_samples = []
with open("data/financial_train.jsonl") as f:
    for line in f:
        raw_samples.append(json.loads(line))

formatted = [format_sample(s) for s in raw_samples]
dataset = Dataset.from_list(formatted)

# 90/10 train-val split
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
val_dataset   = split["test"]

print(f"\n   Train samples : {len(train_dataset)}")
print(f"   Val samples   : {len(val_dataset)}")
print(f"\n   Formatted example (first 300 chars):")
print(train_dataset[0]["text"][:300])

Loading model: Qwen/Qwen2.5-3B-Instruct
This will download ~2GB on first run — cached on subsequent runs.



config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ Model loaded.
   Parameters  : ~3B
   VRAM usage  : ~2.5GB (NF4 4-bit)

   Train samples : 27
   Val samples   : 3

   Formatted example (first 300 chars):
<|im_start|>system
You are a specialized financial analyst with deep expertise in stock market sentiment analysis, ticker extraction, earnings interpretation, and gold-macroeconomic relationships including DXY, real yields, CPI, and central bank dynamics.<|im_end|>
<|im_start|>user
Extract all finan


In [8]:
# ── Phase 3b: Apply LoRA adapters ───────────────────
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Step 1: prepare model for quantized training
model = prepare_model_for_kbit_training(model)

# Step 2: apply LoRA on top
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA applied")
print(f"   Trainable : {trainable:,} ({100*trainable/total:.2f}%)")
print(f"   Frozen    : {total-trainable:,}")

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


✅ LoRA applied
   Trainable : 29,933,568 (1.73%)
   Frozen    : 1,698,672,640


**Phase 4 — LoRA Fine-Tuning**

Purpose: This is the core training phase. LoRA (Low-Rank Adaptation) freezes the 3 billion base model weights and injects tiny trainable adapter matrices into the attention layers. Only ~0.5% of parameters are updated, which is why it fits in Colab's VRAM. The SFTTrainer from the TRL library handles the training loop. On a Colab T4, expect ~8–12 minutes for 3 epochs over 27 training samples. On a real dataset of 1,000+ samples, plan for 45–90 minutes.

In [7]:
# ── Phase 4 FINAL (TRL 0.24.0 compatible) ─────────────────────────────────────
import torch
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir                  = "outputs/financial_lora",
    num_train_epochs            = 5,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-4,
    lr_scheduler_type           = "cosine",
    warmup_steps                = 10,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 5,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    dataloader_num_workers      = 0,
    report_to                   = "none",
    # dataset_text_field, packing, max_seq_length removed — not in TRL 0.24.0 SFTConfig
)

# In TRL 0.24.0, SFTTrainer handles tokenization directly
# We pass the raw text column name here instead
from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(sample):
    return tokenizer(
        sample["text"],
        truncation=True,
        max_length=1024,
        padding=False,
    )

train_tokenized = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
val_tokenized   = val_dataset.map(tokenize_fn,   batched=True, remove_columns=["text"])

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,   # causal LM, not masked LM
)

trainer = SFTTrainer(
    model          = model,
    args           = training_args,
    train_dataset  = train_tokenized,
    eval_dataset   = val_tokenized,
    data_collator  = data_collator,
)

print("🚀 Starting fine-tuning...")
print("   Loss should drop from ~2.0 toward ~0.5\n")

result = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Final training loss : {result.training_loss:.4f}")
print(f"   Total steps         : {result.global_step}")
print(f"   Runtime             : {result.metrics.get('train_runtime', 0):.0f}s")

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/27 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


🚀 Starting fine-tuning...
   Loss should drop from ~2.0 toward ~0.5



Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,No log,2.559093,1.758258,8936.000000,0.497288
2,2.769241,2.090013,2.208606,17872.000000,0.542495
3,2.196741,1.775145,2.212294,26808.000000,0.596745
4,1.670765,1.579299,1.758259,35744.000000,0.638336
5,1.371921,1.549497,1.643225,44680.000000,0.639241



✅ Training complete!
   Final training loss : 2.0022
   Total steps         : 20
   Runtime             : 676s


**Phase 5 — Inference & Testing**

Purpose: Before saving anything, we test the fine-tuned model on three prompt types it was trained on. Compare these outputs to what the base model (before training) would produce — the fine-tuned model should give more structured, financially-precise responses with the exact output format we defined in our dataset. If the output format looks wrong, the training didn't converge properly (go back and check loss values).

In [9]:
# ── Phase 5: Inference & Testing ────────────────────────────────────────
import torch

model = model.to(torch.float16)
model.eval()

def run_inference(instruction: str, input_text: str = "", max_new_tokens: int = 400) -> str:
    system_msg = (
        "You are a specialized financial analyst with deep expertise in "
        "stock market sentiment analysis, ticker extraction, earnings interpretation, "
        "and gold-macroeconomic relationships including DXY, real yields, CPI, and central bank dynamics."
    )
    user_content = instruction
    if input_text.strip():
        user_content += f"\n\n{input_text}"

    messages = [
        {"role": "system",    "content": system_msg},
        {"role": "user",      "content": user_content},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt")
    # Cast input to float16 to match model dtype
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

# ── Test 1: Sentiment Analysis ─────────────────────────────────────────────────
print("=" * 60)
print("TEST 1 — Sentiment Analysis")
print("=" * 60)
result1 = run_inference(
    instruction="Analyze the financial sentiment of the following news headline. Classify as Bullish, Bearish, or Neutral. Provide a brief rationale and extract any tickers or assets mentioned.",
    input_text="Microsoft announces $75B acquisition of gaming studio; deal subject to FTC review. MSFT stock falls 2% on deal premium concerns."
)
print(result1)

# ── Test 2: Gold-Macro Reasoning ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("TEST 2 — Gold-Macro Reasoning")
print("=" * 60)
result2 = run_inference(
    instruction="Given the macroeconomic data snapshot below, predict the directional 30-day outlook for gold (XAU/USD) and explain the three key driving factors. Rate your conviction as High, Medium, or Low.",
    input_text="DXY: 105.1 (+0.9% WoW). US 10Y Real Yield: 1.9%. CPI YoY: 3.8%. VIX: 22.3. Fed Funds Rate: 5.50%. Geopolitical risk: elevated."
)
print(result2)

# ── Test 3: Ticker Extraction ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("TEST 3 — Ticker Extraction & Sentiment Score")
print("=" * 60)
result3 = run_inference(
    instruction="Extract all financial entities from the text below. Return: (1) Stock tickers with company names, (2) Commodities, (3) Macro indicators, (4) Overall market sentiment score from -5 (extreme bear) to +5 (extreme bull).",
    input_text="Amazon reported Q3 AWS revenue of $27.5B, up 19% YoY, beating estimates of $26.1B. Advertising revenue grew 26% to $14.3B. Operating income of $17.4B crushed the $11.6B consensus. AMZN stock surged 8% after hours."
)
print(result3)

print("\n✅ Inference tests complete. Review outputs above for quality.")

TEST 1 — Sentiment Analysis


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


**Sentiment Classification:** **Neutral**

**Rationale:**
The headline discusses a significant corporate action (the $75B acquisition) but does not provide clear indications of investor sentiment towards Microsoft's stock price movement. The mention that MSFT stock fell by 2% due to "deal premium concerns" suggests some negative sentiment among investors, but this is more about short-term market reaction rather than long-term bullish or bearish sentiment.

**Tickers/Assets Mentioned:**
- MSFT (Microsoft Corporation)

This classification is neutral because while there is an acquisition announcement, which could be seen as potentially positive for Microsoft's future prospects, the immediate market reaction (a 2% drop) indicates caution from investors. Without further context or information about the broader market sentiment, it's difficult to classify the overall sentiment as either bullish or bearish.

TEST 2 — Gold-Macro Reasoning
Based on the provided macroeconomic data snapshot, let'

**Phase 6 — Save & Export Adapters**

Purpose: We save only the LoRA adapter weights — not the full 3B model. The adapter is tiny (~50MB) while the base model is ~2GB. This means you can share just the adapter on Hugging Face and anyone with the Qwen2.5-3B base model can use your specialized financial version instantly. We also show how to merge the adapter into the base model for deployment if you want a single standalone file.

In [10]:
# ── Phase 6: Save & Export ────────────────────────────────────────────────────
import os
from google.colab import files

ADAPTER_PATH = "outputs/financial_lora_final"
MERGED_PATH  = "outputs/financial_lora_merged"

os.makedirs(ADAPTER_PATH, exist_ok=True)

# ── Save LoRA adapters only (~50MB) ───────────────────────────────────────────
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"✅ LoRA adapters saved to: {ADAPTER_PATH}")

adapter_size_mb = sum(
    os.path.getsize(os.path.join(ADAPTER_PATH, f))
    for f in os.listdir(ADAPTER_PATH)
    if os.path.isfile(os.path.join(ADAPTER_PATH, f))
) / 1e6
print(f"   Adapter size: {adapter_size_mb:.1f} MB (vs ~2GB for full base model)")

# ── Merge adapters into base model (for standalone deployment) ────────────────
print("\n⏳ Merging LoRA adapters into base model weights...")
print("   This creates a single deployable model file (~6GB in float16)")
print("   Use this if you want to run via Ollama or llama.cpp locally")

merged_model = model.merge_and_unload()   # fuses adapter weights into base
merged_model.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)
print(f"✅ Merged model saved to: {MERGED_PATH}")

# ── Save adapter config summary ───────────────────────────────────────────────


✅ LoRA adapters saved to: outputs/financial_lora_final
   Adapter size: 71.4 MB (vs ~2GB for full base model)

⏳ Merging LoRA adapters into base model weights...
   This creates a single deployable model file (~6GB in float16)
   Use this if you want to run via Ollama or llama.cpp locally


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


AttributeError: property 'peft_config' of 'PeftModelForCausalLM' object has no deleter

In [12]:
import json, os

# Model already saved
config_summary = {
    "base_model"       : "unsloth/Qwen2.5-3B-Instruct",
    "adapter_path"     : "outputs/financial_lora_final",
    "merged_path"      : "outputs/financial_lora_merged",
    "adapter_size_mb"  : 71.4,
    "lora_rank"        : 16,
    "lora_alpha"       : 32,
    "task_types"       : ["sentiment_classification", "gold_macro_reasoning", "ticker_extraction"],
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/training_config.json", "w") as f:
    json.dump(config_summary, f, indent=2)

print("✅ Project complete! Summary:")
print(json.dumps(config_summary, indent=2))

print("\n Files ready in Colab:")
print("   outputs/financial_lora_final/   ← 71.4MB adapter (download this)")
print("   outputs/financial_lora_merged/  ← full merged model for Ollama")

# ── Download the adapter ──────────────────────────────────────────────────────
import shutil
from google.colab import files

shutil.make_archive("financial_lora_adapter", "zip", "outputs/financial_lora_final")
files.download("financial_lora_adapter.zip")
print("\n⬇️  Adapter download started — check your browser downloads folder.")

✅ Project complete! Summary:
{
  "base_model": "unsloth/Qwen2.5-3B-Instruct",
  "adapter_path": "outputs/financial_lora_final",
  "merged_path": "outputs/financial_lora_merged",
  "adapter_size_mb": 71.4,
  "lora_rank": 16,
  "lora_alpha": 32,
  "task_types": [
    "sentiment_classification",
    "gold_macro_reasoning",
    "ticker_extraction"
  ]
}

 Files ready in Colab:
   outputs/financial_lora_final/   ← 71.4MB adapter (download this)
   outputs/financial_lora_merged/  ← full merged model for Ollama


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


⬇️  Adapter download started — check your browser downloads folder.


# Save merged model in Colab

In [13]:
# ── Save Merged Model in float16 (for HuggingFace + GGUF conversion) ──────────
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MERGED_PATH = "outputs/financial_lora_merged"
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

print("Loading base model in float16 for clean merge...")

# we need to load base model fresh in float16 (not 4bit — merging needs full precision)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

# need to load our LoRA adapter on top
print("Applying LoRA adapter...")
peft_model = PeftModel.from_pretrained(
    base_model,
    "outputs/financial_lora_final",  # your saved adapter path
)

# need to merge and unload — fuses adapter into base weights
print("Merging adapter into base model...")
merged = peft_model.merge_and_unload()

# need to ave merged model in float16
merged.save_pretrained(MERGED_PATH, safe_serialization=True)
tokenizer.save_pretrained(MERGED_PATH)

print(f"✅ Merged model saved to: {MERGED_PATH}")
print("   Format: float16 safetensors (~6GB)")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model in float16 for clean merge...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Applying LoRA adapter...
Merging adapter into base model...


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.we

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Merged model saved to: outputs/financial_lora_merged
   Format: float16 safetensors (~6GB)


# Download merged model from Colab

In [14]:
# ── Download merged model ──────────────────────────────────────────────────────
# ~6GB zip — takes 10-20 minutes to download
import shutil
from google.colab import files

print("Zipping merged model...")
shutil.make_archive("financial_llm_merged", "zip", "outputs/financial_lora_merged")
files.download("financial_llm_merged.zip")
print("✅ Download started.")

Zipping merged model...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started.


# Putting the model in HuggingFace Hub

In [26]:
from huggingface_hub import HfApi

TOKEN    = "..."
USERNAME = "mohammadbk321"
REPO     = f"{USERNAME}/financial-qwen-3b"

api = HfApi()

api.create_repo(
    repo_id   = REPO,
    repo_type = "model",
    private   = True,
    exist_ok  = True,
    token     = TOKEN,
)
print("✅ Repo created")

api.upload_folder(
    folder_path = "outputs/financial_lora_merged",
    repo_id     = REPO,
    repo_type   = "model",
    token       = TOKEN,
)
print(f"✅ Uploaded: huggingface.co/{REPO}")

✅ Repo created
✅ Uploaded: huggingface.co/mohammadbk321/financial-qwen-3b
